# El Reto Kafka

## Instalación

In [505]:
# %pip install -q transformers torch pandas

## Constantes y Librerias

In [506]:
import re
from contextlib import redirect_stdout
from pathlib import Path
from tabulate import tabulate
import torch
from transformers import AutoTokenizer, AutoModel

In [507]:
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
CORPUS_PATH = Path("data/corpus.txt")
MODEL_NAME = "bert-base-multilingual-cased"
SENTENCES_LIMIT = 1
TOP_K = 5

# Corta donde un punto, cierre de interrogación o de exclamación va seguido
# del arranque de una oración nueva: mayúscula, signo de apertura o raya de
# diálogo. Tolera un signo de cierre (comilla o paréntesis) antes del espacio,
# para no partir dentro de una cita.
SENTENCE_PATTERN = re.compile(
    r'(?:(?<=[.!?…])|(?<=[.!?…][»")\]]))'
    r'\s+'
    r'(?=[«¡¿—"(\[]*[A-ZÁÉÍÓÚÜÑ])'
)

## Funciones auxiliares

In [508]:
def getModel(model_name, model_dir):
    local_path = model_dir / model_name
    config_file = local_path / "config.json"

    # Se valida contra un archivo concreto y no contra la carpeta: si la
    # descarga se interrumpe, el directorio queda creado pero incompleto y
    # la caché daría un falso positivo en la ejecución siguiente.
    if not config_file.exists():
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)

        # save_pretrained crea el directorio si hace falta.
        tokenizer.save_pretrained(local_path)
        model.save_pretrained(local_path)

    return local_path

In [509]:
def getTokenizer(model_path):
    return AutoTokenizer.from_pretrained(model_path)

In [510]:
def loadModel(model_path):
    model = AutoModel.from_pretrained(
        model_path,
        output_attentions=True
    )
    model.eval()
    return model

In [511]:
def getCorpus(corpus_path):
    with open(corpus_path, "r", encoding="utf-8") as file:
        raw_text = file.read()

    # El archivo conserva los saltos de línea de la maquetación del libro,
    # que cortan las oraciones a media cláusula. Se unen en un texto continuo
    # y se normalizan los espacios antes de segmentar.
    text = re.sub(r"\s+", " ", raw_text).strip()

    return [
        sentence.strip()
        for sentence in SENTENCE_PATTERN.split(text)
        if sentence.strip()
    ]

In [512]:
def getTokens(sentence, tokenizer):
    encoded = tokenizer(sentence, add_special_tokens=True)
    return tokenizer.convert_ids_to_tokens(encoded["input_ids"])

In [513]:
def tokenizeCorpus(corpus, tokenizer):
    return [
        {
            "sentence": sentence,
            "tokens": getTokens(sentence, tokenizer)
        }
        for sentence in corpus
    ]

In [514]:
def saveResult(function, log_name, log_dir):
    log_path = Path(log_dir) / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)

    with open(log_path, "w", encoding="utf-8") as file:
        with redirect_stdout(file):
            function()

    return log_path

In [515]:
model_path = getModel(model_name=MODEL_NAME, model_dir=MODEL_DIR)

tokenizer = getTokenizer(model_path)
model = loadModel(model_path)

corpus = getCorpus(corpus_path=CORPUS_PATH)

tokenized_corpus = tokenizeCorpus(corpus, tokenizer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Parte A 

In [516]:
def printTokenizedCorpus(tokenized_corpus):
    for item in tokenized_corpus:
        print("Oración:", item["sentence"])
        print("Tokens:", item["tokens"])
        print()

In [517]:
saveResult(
    lambda: printTokenizedCorpus(tokenized_corpus),
    log_name="tokenization.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/tokenization.txt')

In [518]:
printTokenizedCorpus(tokenized_corpus[:SENTENCES_LIMIT])

Oración: De no haberle parecido oportuna tal medida, ella misma hubiera quitado la sábana, pues fácil era comprender que, para Gregorio, el aislarse no era nada agradable.
Tokens: ['[CLS]', 'De', 'no', 'haber', '##le', 'parecido', 'op', '##ort', '##una', 'tal', 'medida', ',', 'ella', 'misma', 'hubiera', 'quit', '##ado', 'la', 's', '##ában', '##a', ',', 'pues', 'fácil', 'era', 'comprende', '##r', 'que', ',', 'para', 'Gregorio', ',', 'el', 'ai', '##slar', '##se', 'no', 'era', 'nada', 'ag', '##rada', '##ble', '.', '[SEP]']



## Parte B

In [519]:
def getAttentions(sentence, tokenizer, model):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        add_special_tokens=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.attentions

In [520]:
def printAttentionShapes(corpus, tokenizer, model):
    for sentence in corpus:
        attentions = getAttentions(sentence, tokenizer, model)

        print("Oración:", sentence)
        print("Número de capas:", len(attentions))

        for i, attention in enumerate(attentions):
            print(
                f"Capa {i}:",
                tuple(attention.shape)
            )

        print()

In [521]:
saveResult(
    lambda: printAttentionShapes(corpus, tokenizer, model),
    log_name="attention_shapes.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_shapes.txt')

In [522]:
printAttentionShapes(corpus[:SENTENCES_LIMIT], tokenizer, model)

Oración: De no haberle parecido oportuna tal medida, ella misma hubiera quitado la sábana, pues fácil era comprender que, para Gregorio, el aislarse no era nada agradable.
Número de capas: 12
Capa 0: (1, 12, 44, 44)
Capa 1: (1, 12, 44, 44)
Capa 2: (1, 12, 44, 44)
Capa 3: (1, 12, 44, 44)
Capa 4: (1, 12, 44, 44)
Capa 5: (1, 12, 44, 44)
Capa 6: (1, 12, 44, 44)
Capa 7: (1, 12, 44, 44)
Capa 8: (1, 12, 44, 44)
Capa 9: (1, 12, 44, 44)
Capa 10: (1, 12, 44, 44)
Capa 11: (1, 12, 44, 44)



## Parte C

In [523]:
def getTopAttention(sentence, target_token, layer, head, tokenizer, model, top_n, occurrence=0):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        add_special_tokens=True
    )

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    with torch.no_grad():
        outputs = model(**inputs)

    attentions = outputs.attentions

    # Una palabra repetida ocupa varias filas distintas de la matriz, y cada
    # una tiene su propia representación contextual. No basta con localizar
    # el token: hay que declarar cuál de sus ocurrencias se analiza.
    occurrences = [
        index
        for index, token in enumerate(tokens)
        if token == target_token
    ]

    if not occurrences:
        raise ValueError(
            f"El token '{target_token}' no existe en: {tokens}"
        )

    if occurrence >= len(occurrences):
        raise ValueError(
            f"El token '{target_token}' aparece en los índices {occurrences}. "
            f"No existe la ocurrencia {occurrence}."
        )

    token_index = occurrences[occurrence]

    # attention[layer]:
    # (batch, heads, tokens, tokens)
    #
    # Seleccionamos:
    # batch 0
    # cabeza indicada
    # fila del token que estamos analizando
    weights = attentions[layer][0, head, token_index]

    top_values, top_indices = torch.topk(
        weights,
        k=min(top_n, len(tokens))
    )

    rows = []

    for position, (index, weight) in enumerate(
        zip(top_indices.tolist(), top_values.tolist()),
        start=1
    ):
        rows.append([
            position,
            tokens[index],
            index,
            weight
        ])

    return {
        "token_index": token_index,
        "occurrences": occurrences,
        "rows": rows
    }

In [524]:
def printTopAttention(sentence, target_token, layer, head, tokenizer, model, top_n, occurrence=0):
    result = getTopAttention(
        sentence=sentence,
        target_token=target_token,
        layer=layer,
        head=head,
        tokenizer=tokenizer,
        model=model,
        top_n=top_n,
        occurrence=occurrence
    )

    print("Oración:", sentence)
    print("Token analizado:", target_token)
    print("Ocurrencias del token:", result["occurrences"])
    print("Índice analizado:", result["token_index"])
    print("Capa:", layer)
    print("Cabeza:", head)

    print(
        tabulate(
            result["rows"],
            headers=[
                "Ranking",
                "Token",
                "Índice",
                "Atención"
            ],
            tablefmt="grid",
            floatfmt=".6f"
        )
    )

    print()

In [525]:
def printAttentionAnalysis(sentence, target_tokens, layers, heads, tokenizer, model,
                           top_n, occurrence=0):
    print("=" * 80)
    print("Oración:", sentence)
    print("Tokens:", getTokens(sentence, tokenizer))
    print("=" * 80)
    print()

    for target_token in target_tokens:
        for layer in layers:
            for head in heads:

                printTopAttention(
                    sentence=sentence,
                    target_token=target_token,
                    layer=layer,
                    head=head,
                    tokenizer=tokenizer,
                    model=model,
                    top_n=top_n,
                    occurrence=occurrence
                )

In [526]:
# corpus[5]: "La madre había querido visitar a Gregorio enseguida, pero el
# padre y la hermana la habían hecho desistir con argumentos que Gregorio
# escuchó con la mayor atención y aprobó por entero." (44 tokens)
sentence = corpus[5]

target_tokens = [
    "madre",
    "Gregorio"
]

layers = [
    2,
    8
]

heads = [
    0,
    5
]

In [527]:
saveResult(
    lambda: printAttentionAnalysis(
        sentence=sentence,
        target_tokens=target_tokens,
        layers=layers,
        heads=heads,
        tokenizer=tokenizer,
        model=model,
        top_n=TOP_K
    ),
    log_name="attention_analysis.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_analysis.txt')

In [528]:
printAttentionAnalysis(
    sentence=sentence,
    target_tokens=target_tokens,
    layers=layers,
    heads=heads,
    tokenizer=tokenizer,
    model=model,
    top_n=TOP_K
)

Oración: La madre había querido visitar a Gregorio enseguida, pero el padre y la hermana la habían hecho desistir con argumentos que Gregorio escuchó con la mayor atención y aprobó por entero.
Tokens: ['[CLS]', 'La', 'madre', 'había', 'quer', '##ido', 'visitar', 'a', 'Gregorio', 'ens', '##egu', '##ida', ',', 'pero', 'el', 'padre', 'y', 'la', 'hermana', 'la', 'habían', 'hecho', 'des', '##istir', 'con', 'argumento', '##s', 'que', 'Gregorio', 'es', '##cu', '##chó', 'con', 'la', 'mayor', 'atención', 'y', 'ap', '##robó', 'por', 'enter', '##o', '.', '[SEP]']

Oración: La madre había querido visitar a Gregorio enseguida, pero el padre y la hermana la habían hecho desistir con argumentos que Gregorio escuchó con la mayor atención y aprobó por entero.
Token analizado: madre
Ocurrencias del token: [2]
Índice analizado: 2
Capa: 2
Cabeza: 0
+-----------+---------+----------+------------+
|   Ranking | Token   |   Índice |   Atención |
+===========+=========+==========+============+
|         1 | [

## Parte D

In [529]:
def compareAttention(sentence_1, sentence_2, target_token, layer, head, tokenizer, model,
                     top_n, occurrence_1=0, occurrence_2=0):
    result_1 = getTopAttention(
        sentence_1,
        target_token,
        layer,
        head,
        tokenizer,
        model,
        top_n,
        occurrence_1
    )

    result_2 = getTopAttention(
        sentence_2,
        target_token,
        layer,
        head,
        tokenizer,
        model,
        top_n,
        occurrence_2
    )

    print("=" * 80)
    print(f"COMPARACIÓN DEL TOKEN: {target_token}")
    print(f"Capa: {layer} | Cabeza: {head}")
    print("=" * 80)

    print("\nORACIÓN 1:")
    print(sentence_1)
    print("Ocurrencias del token:", result_1["occurrences"])
    print("Índice analizado:", result_1["token_index"])
    print(tabulate(
        result_1["rows"],
        headers=["Ranking", "Token", "Índice", "Atención"],
        tablefmt="grid",
        floatfmt=".6f"
    ))

    print("\nORACIÓN 2:")
    print(sentence_2)
    print("Ocurrencias del token:", result_2["occurrences"])
    print("Índice analizado:", result_2["token_index"])
    print(tabulate(
        result_2["rows"],
        headers=["Ranking", "Token", "Índice", "Atención"],
        tablefmt="grid",
        floatfmt=".6f"
    ))

In [530]:
# Comparación 1: "Gregorio" en dos funciones sintácticas distintas.
# corpus[5]  -> "... visitar a Gregorio ..."  objeto, marcado por la "a"
# corpus[21] -> "Gregorio había bajado ..."   sujeto, en posición inicial
gregorio_sentence_1 = corpus[5]
gregorio_sentence_2 = corpus[21]

In [531]:
saveResult(
    lambda: compareAttention(
        sentence_1=gregorio_sentence_1,
        sentence_2=gregorio_sentence_2,
        target_token="Gregorio",
        layer=8,
        head=5,
        tokenizer=tokenizer,
        model=model,
        top_n=TOP_K
    ),
    log_name="attention_comparison_gregorio.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_comparison_gregorio.txt')

In [532]:
compareAttention(
    sentence_1=gregorio_sentence_1,
    sentence_2=gregorio_sentence_2,
    target_token="Gregorio",
    layer=8,
    head=5,
    tokenizer=tokenizer,
    model=model,
    top_n=TOP_K
)

COMPARACIÓN DEL TOKEN: Gregorio
Capa: 8 | Cabeza: 5

ORACIÓN 1:
La madre había querido visitar a Gregorio enseguida, pero el padre y la hermana la habían hecho desistir con argumentos que Gregorio escuchó con la mayor atención y aprobó por entero.
Ocurrencias del token: [8, 28]
Índice analizado: 8
+-----------+----------+----------+------------+
|   Ranking | Token    |   Índice |   Atención |
+===========+==========+==========+============+
|         1 | [SEP]    |       43 |   0.418983 |
+-----------+----------+----------+------------+
|         2 | [CLS]    |        0 |   0.335413 |
+-----------+----------+----------+------------+
|         3 | Gregorio |        8 |   0.146193 |
+-----------+----------+----------+------------+
|         4 | Gregorio |       28 |   0.055967 |
+-----------+----------+----------+------------+
|         5 | .        |       42 |   0.028545 |
+-----------+----------+----------+------------+

ORACIÓN 2:
Gregorio había bajado la sábana más que de costumbre

In [533]:
# Comparación 2: "madre" con y sin otros sustantivos de parentesco alrededor.
# Se mantiene la misma capa y la misma cabeza que en la comparación 1, así que
# lo único que cambia es el token desde el que se mira y el contexto.
#
# En las dos oraciones "madre" ocupa el índice 2 y es sujeto precedido de "La",
# de modo que la posición y la función sintáctica se mantienen constantes.
#
# corpus[5]:  "La madre ... pero el padre y la hermana ..."  (44 tokens, con parentescos)
# corpus[19]: "La madre acudió eufórica, ..."                (20 tokens, sin parentescos)
madre_sentence_1 = corpus[5]
madre_sentence_2 = corpus[19]

In [534]:
saveResult(
    lambda: compareAttention(
        sentence_1=madre_sentence_1,
        sentence_2=madre_sentence_2,
        target_token="madre",
        layer=8,
        head=5,
        tokenizer=tokenizer,
        model=model,
        top_n=TOP_K
    ),
    log_name="attention_comparison_madre.txt",
    log_dir=RESULTS_DIR
)

WindowsPath('results/attention_comparison_madre.txt')

In [535]:
compareAttention(
    sentence_1=madre_sentence_1,
    sentence_2=madre_sentence_2,
    target_token="madre",
    layer=8,
    head=5,
    tokenizer=tokenizer,
    model=model,
    top_n=TOP_K
)

COMPARACIÓN DEL TOKEN: madre
Capa: 8 | Cabeza: 5

ORACIÓN 1:
La madre había querido visitar a Gregorio enseguida, pero el padre y la hermana la habían hecho desistir con argumentos que Gregorio escuchó con la mayor atención y aprobó por entero.
Ocurrencias del token: [2]
Índice analizado: 2
+-----------+---------+----------+------------+
|   Ranking | Token   |   Índice |   Atención |
+===========+=========+==========+============+
|         1 | padre   |       15 |   0.374190 |
+-----------+---------+----------+------------+
|         2 | hermana |       18 |   0.373846 |
+-----------+---------+----------+------------+
|         3 | [SEP]   |       43 |   0.084659 |
+-----------+---------+----------+------------+
|         4 | madre   |        2 |   0.071066 |
+-----------+---------+----------+------------+
|         5 | [CLS]   |        0 |   0.038377 |
+-----------+---------+----------+------------+

ORACIÓN 2:
La madre acudió eufórica, pero se quedó muda al llegar a la puerta.
Ocur